# HelioAI — the St Patrick's Day storm, 17 March 2015

The largest geomagnetic storm of solar cycle 24 (Dst minimum −223 nT), worked end to end:
find the parameters, download them, characterise the shock and the driver, and compute the
plasma regimes on either side.

This notebook is a **worked scientific example** rather than a feature tour — see
`01_jupyter_tour.ipynb` for that.

**Why this event, and not the 2003 Halloween storm**

Halloween 2003 is the more famous choice, and it was this notebook's first subject. It was
replaced because the archives cannot support it. Measured over 2003-10-28T20:00 →
2003-10-30T06:00: ACE/SWEPAM density, speed and temperature are **100 % fill values** — the
instrument saturated in the particle storm — Wind/SWE is 69–74 % fill, OMNI 1-min is 57 %
fill and its temperature 99.7 %, and SOHO/CELIAS, the one product that returns clean data,
clips its speed at a constant 1019 km/s, its instrument ceiling. No standard speasy product
reproduces that event's headline numbers, which is a large part of why it is famous. It
makes a poor first demonstration.

17 March 2015 has clean science-quality coverage from a single spacecraft, so every number
below is reproducible.

**All reference values here were measured from the archive, not quoted.** The script that
measures them is [`verify_reference_values.py`](verify_reference_values.py) — run it and
compare. Where a published value and the archive disagreed, the archive won. If your run
disagrees with a number in this notebook, the number in this notebook is wrong; please open
an issue.

**Before running this**

1. `pip install helioai` (or `uv sync` from a clone)
2. One LLM provider key in `.env` — see the
   [installation guide](https://erdoganfurkan.github.io/HelioAI/installation/)
3. `helioai index` once (~10 min)

> Outputs are stripped on purpose — run the cells to produce your own. Wall time is
> dominated by data download; expect a few minutes.


In [ ]:
%load_ext helioai.interfaces.jupyter_magic

import os

provider = os.environ.get("HELIOAI_LLM_PROVIDER", "azure")
print(f"LLM provider : {provider}")
print("Ready — run cells below with Shift+Enter.")

---
## Act I — Data Discovery

Before any analysis, you need correct speasy parameter IDs. The same physical quantity appears under dozens of names across ACE, Wind, DSCOVR, STEREO…  
Classic pitfalls: `onboard` vs `prime` moments, `RTN` vs `GSE` frame, `HIA` vs `CODIF` for ions, `MFI` vs `MAG` for magnetometers.

One pitfall matters more than the rest here, and the prompt below names it explicitly rather than hoping the agent guesses: **`K0` products are browse quality, not science quality.** `WI_K0_SWE/Np` passes every fill-value check for this event and still carries single-point spikes to 166 cm⁻³ — enough to put a false shock four hours before the real one and return a compression ratio of 1.1. `WI_H1_SWE/*_moment` is the science product, and it is clean.

The `parameter_hunter` sub-agent searches the 83 000-entry RAG index and returns the canonical IDs with caveats.

In [ ]:
%%helioai
I want to analyze the St Patrick's Day geomagnetic storm of 17 March
2015 (Dst minimum -223 nT), driven by a CME-driven shock that reached
L1 early on the 17th. My event window is 2015-03-16T18:00:00 to
2015-03-18T12:00:00 UT.

I want the whole analysis from ONE spacecraft so the shock jump
conditions are self-consistent. Use Wind:
  1. IMF vector in GSM from Wind/MFI at the highest available cadence
  2. Proton number density from Wind/SWE
  3. Proton bulk speed from Wind/SWE
  4. Proton thermal speed or temperature from Wind/SWE

Important: give me the SCIENCE-quality SWE product, not the K0 browse
product. K0 carries single-point spikes that survive fill-value
filtering and will wreck the shock analysis.

Then, separately, the IMF vector in GSM from ACE/MAG -- I will compare
shock arrival times between the two spacecraft in a later step.

For each parameter give me the exact speasy ID, the native cadence, and
the actual fraction of valid samples in this window. Check the coverage
rather than assuming it: report the fill-value fraction you measure.

---
## Act II — Solar Wind Visualization

A fast forward shock crossed Wind at **04:01 UT on 17 March**, followed by a compressed
sheath, then the prolonged southward-Bz interval that drove the main storm phase around
midday.

The shock is easy to get wrong. A density jump alone is not a shock — there is a larger
density rise at 12:51 UT with no speed change at all, which is a compression structure
inside the driver. A fast forward shock requires **n, |B| and V to jump together**, and that
criterion picks 04:01 UT.

**Reference values, all measured from the archive over this window** — reproduce them with
`python examples/verify_reference_values.py`:

| Quantity | Measured | Product |
|:---|:---|:---|
| Shock crossing (n, &#124;B&#124;, V together) | 04:00:59 UT Mar 17 | `WI_H1_SWE` + `WI_H0_MFI` |
| Shock arrival at Wind, from &#124;B&#124; | 04:01:30 UT Mar 17 | `WI_H0_MFI/BGSM` (60 s) |
| Shock arrival at ACE, from &#124;B&#124; | 04:04:25 UT Mar 17 | `AC_H0_MFI/BGSM` (16 s) |
| Upstream speed (1 h before) | 411.3 km/s | `WI_H1_SWE/Proton_V_moment` |
| Downstream speed (30 min after) | 514.1 km/s | idem |
| Peak speed in window | 657.4 km/s | idem |
| Upstream density | 17.43 cm⁻³ | `WI_H1_SWE/Proton_Np_moment` |
| Downstream density | 45.12 cm⁻³ | idem |
| Peak density in window | 57.9 cm⁻³ | idem |
| Upstream &#124;B&#124; | 10.00 nT | `WI_H0_MFI/BGSM` |
| Downstream &#124;B&#124; | 25.27 nT | idem |
| Max &#124;B&#124; on Mar 17 | 33.11 nT | idem |
| Min Bz GSM on Mar 17 | −26.05 nT at 12:19:30 UT | idem |

> Coverage in this window: Wind SWE H1 moments **0.0 %** invalid, ACE MAG 0.1 %, Wind MFI
> 2.4 % (counting a vector sample as lost if any component is). Nothing here needs
> gap-filling — which is the whole reason this event replaced Halloween 2003.

In [ ]:
%%helioai
Using the Wind parameter IDs you just identified, download and plot the
St Patrick's Day storm solar wind for 2015-03-16T18:00:00 to
2015-03-18T12:00:00 UT.

Make a four-panel time series figure (dark background, shared x-axis):
  Panel 1 -- IMF |B| (white) and Bz in GSM (blue when > 0, red when < 0)
             draw a thin grey dashed line at Bz = 0
  Panel 2 -- Proton number density (cm^-3)
  Panel 3 -- Proton bulk speed (km/s)
  Panel 4 -- Proton temperature (convert from thermal speed if needed)

On each panel:
  - Mark the shock arrival with a vertical orange line. Find it from the
    data: require n, |B| AND V to increase together across the jump.
    A density jump on its own is not a shock -- there is a bigger one
    near 12:51 UT with no speed change, and it is not the shock.
  - Shade the sheath region (between shock and the first sustained
    southward-Bz interval) in semi-transparent orange
  - Mark the onset of the main southward-Bz driving interval with a
    dashed vertical red line labelled 'storm main phase onset'

After the figure, print a summary table with:
  - Detected shock time (UT), and the n, |B|, V jumps that identified it
  - Upstream (1-hour average before shock): |B|, n, V, T
  - Downstream (30-min average after shock): |B|, n, V, T
  - Min Bz and its time
  - The fraction of invalid samples you had to discard per parameter

---
## Act III — Rankine-Hugoniot Shock Physics

The Rankine-Hugoniot jump conditions for a fast-mode MHD shock constrain the compression ratio, Mach number, and downstream temperature.  
For a strong shock in a γ = 5/3 gas, the theoretical maximum compression is:

$$r_{\max} = \frac{\gamma+1}{\gamma-1} = 4$$

The Alfvénic Mach number $M_A = V_{\mathrm{shock}} / V_A^{\mathrm{upstream}}$ classifies shock strength.

**Expected, measured from the archive at the 04:01 UT crossing:**

| Quantity | Measured | Implication |
|:---|:---|:---|
| Density compression r_n | 2.59 | Strong, but below the r = 4 limit |
| B compression r_B | 2.53 | Agrees with r_n — a good consistency check |
| V_A upstream | 52.3 km/s | Low-β solar wind |
| c_s upstream | 36.5 km/s | T₁ = 96.8 kK |
| V_shock (de Hoffmann-Teller) | ~838 km/s | Approximation, see below |
| M_A | ~16 | Strong CME-driven shock |
| T upstream → downstream | 96.8 → 250.0 kK | Rankine-Hugoniot heating |

> **r_n ≈ r_B is the check that matters.** Two independently measured compressions landing on
> 2.59 and 2.53 says the jump is real and the averaging windows are sensible. When they
> disagree by more than ~20 %, suspect the shock time before suspecting the physics.

> The de Hoffmann-Teller estimate of $V_{\text{shock}}$ is a one-line approximation, and Act IV
> gives an independent number from two-spacecraft timing. They will not match exactly —
> that disagreement is the interesting part, not an error.

In [ ]:
%%helioai
Using the upstream/downstream averages you computed at the St Patrick's
Day storm shock, perform a Rankine-Hugoniot analysis:

1. Density compression ratio  r_n = n2 / n1
2. Magnetic field compression r_B = |B|2 / |B|1
3. Check r_n against r_B -- they should agree to within about 20% for a
   real shock with sensible averaging windows. If they do not, say so
   and reconsider the shock time before trusting the rest.
4. Upstream Alfven speed V_A1 = |B|1 / sqrt(mu0 * m_p * n1)
5. Estimate the shock speed V_shock using the de Hoffmann-Teller
   approximation: V_shock = V2 * r_n / (r_n - 1)
6. Alfvenic Mach number M_A = V_shock / V_A1
7. Compare r_n to the theoretical strong-shock limit (gamma+1)/(gamma-1)
   for gamma = 5/3 -- how close to the limit is this shock?
8. Compute the expected downstream temperature from Rankine-Hugoniot:
   T2 = T1 * [2*gamma*Ms^2 - (gamma-1)] * [(gamma-1)*Ms^2 + 2]
                / [(gamma+1)^2 * Ms^2]
   where Ms is the sonic Mach number (use c_s1 = sqrt(gamma*k_B*T1/m_p))
   and compare to the observed T2

Conclude with a one-paragraph physical interpretation: shock strength,
implications for solar energetic particle acceleration via diffusive
shock acceleration (DSA), and whether the observed compression is
consistent with an MHD description or hints at non-MHD effects.

Be explicit about which numbers are measured and which come from an
approximate formula.

---
## Act IV — Multi-spacecraft Shock Timing

In March 2015 both ACE and Wind sat near L1, tens of R_E apart. The arrival time difference $\Delta t$ between them, combined with their separation vector $\mathbf{\Delta r}$, constrains how the shock front was oriented:

$$\hat{n} \parallel \frac{\mathbf{\Delta r}_{12}}{V_{\text{shock}} \cdot \Delta t_{12}}$$

**Measured positions at shock arrival** — from `amda/ace_xyz_gse` and `amda/wnd_xyz_gse`, in R_E. The prompt asks the agent to fetch these rather than take them from this table:

| | X | Y | Z |
|:---|---:|---:|---:|
| ACE | 221.5 | −10.8 | −23.4 |
| Wind | 253.1 | 54.5 | 12.6 |
| ΔR (ACE − Wind) | −31.6 | −65.2 | −36.0 |

Wind is **31.6 R_E further sunward**, so it should see the shock first. Measured lag: **+175 s**, ACE behind Wind — the right sign, which is the first thing to check.

**The interesting part.** Dividing |ΔX| alone by Δt gives **1150 km/s**, well above the ~838 km/s from Act III. A shock front travelling straight down the Sun–Earth line would give the same number both ways. The excess means the front is **tilted**: it sweeps across the ACE–Wind separation faster in X than it actually propagates. Two independent estimates disagreeing by ~37 % is the physics here, not a mistake — reconciling them is the exercise, and two spacecraft are genuinely not enough to pin the normal down.

In [ ]:
%%helioai
Compare the St Patrick's Day storm shock arrival at ACE and Wind.
Time window: 2015-03-17T03:30:00 to 2015-03-17T05:00:00 UT.

Fetch IMF |B| in GSM from both spacecraft at native cadence and overlay
them on one plot (different colours per spacecraft, legend with names).
Use |B| rather than density: both magnetometers have essentially full
coverage here, and the shock is a sharp field jump.

Then:
1. Measure the shock arrival time at each spacecraft from the |B| jump,
   and the timing lag delta_t. Which spacecraft sees it first, and does
   that match their X positions?
2. Fetch the actual GSE positions of ACE and Wind at shock arrival --
   do not assume them. Compute the separation vector dR in km.
3. Divide the X separation by delta_t. Compare that apparent speed to
   the V_shock you estimated in Act III from the jump conditions.
4. The two numbers will not agree. Explain why: what does an apparent
   X-speed HIGHER than the propagation speed tell you about the tilt of
   the shock front relative to the Sun-Earth line?
5. Give a rough shock normal direction in GSE, and state honestly how
   well two spacecraft can constrain it -- what would you need to do
   better?

---
## Act V — Instant Plasma Physics Reference

For quick sanity checks, call PlasmaPy tools **directly** — zero latency, no LLM call, no API key needed.  
Useful before an agent session to define expected value ranges, or after to verify the agent's numbers.

In [ ]:
from helioai.tools.plasmapy_tools import (
    alfven_speed,
    debye_length,
    gyrofrequency,
    inertial_length,
    plasma_beta,
)

# St Patrick's Day 2015 -- three regimes, each a measured Wind average over the
# interval named in the label (MFI for B, SWE H1 moments for n and T).
regions = {
    "Quiet upstream   (Mar 16 22-23 UT)": {"B_nT": 8.3, "n_cm3": 17.7, "T_eV": 9.9},
    "Shocked sheath   (Mar 17 04-05 UT)": {"B_nT": 24.4, "n_cm3": 43.8, "T_eV": 21.0},
    "Southward-Bz driver (Mar 17 12-14 UT)": {"B_nT": 30.7, "n_cm3": 24.8, "T_eV": 32.2},
}

hdr = f"{'Region':<38}  {'beta':>5}  {'VA km/s':>9}  {'fci Hz':>8}  {'lD m':>7}  {'di km':>7}"
print(hdr)
print("-" * len(hdr))

results = {}
for label, p in regions.items():
    b = await plasma_beta(p["B_nT"], p["n_cm3"], p["T_eV"])
    va = await alfven_speed(p["B_nT"], p["n_cm3"])
    fci = await gyrofrequency(p["B_nT"], "proton")
    ld = await debye_length(p["n_cm3"], p["T_eV"])
    di = await inertial_length(p["n_cm3"], "proton")
    results[label] = {"b": b, "va": va}
    print(
        f"{label:<38}  {b['beta']:>5.2f}  "
        f"{va['alfven_speed_km_s']:>9.1f}  "
        f"{fci['frequency_Hz']:>8.4f}  "
        f"{ld['debye_length_m']:>7.2f}  "
        f"{di['inertial_length_km']:>7.1f}"
    )

print()
up_key = "Quiet upstream   (Mar 16 22-23 UT)"
sheath_key = "Shocked sheath   (Mar 17 04-05 UT)"
ups, down = regions[up_key], regions[sheath_key]
va1 = results[up_key]["va"]["alfven_speed_km_s"]
print(f"Density compression n_sheath/n_up : {down['n_cm3'] / ups['n_cm3']:.1f}x")
print(f"B-field compression B_sheath/B_up : {down['B_nT'] / ups['B_nT']:.1f}x")
print(f"Upstream V_A                      : {va1:.1f} km/s")
print("Strong-shock limit (g=5/3)        : r_max = 4.0")
print()
print("These are hour-long averages, so they differ from the sharp")
print("upstream/downstream jump in Act III -- expect the same order, not")
print("the same number.")

---
## Act VI — Reproducible Export

Every `run_python` call from the agent is **automatically saved** as a `code_N.py` script alongside the figures in the workspace.  
Ask the agent to bundle the full session into a standalone script — runnable with only `speasy`, `numpy`, `matplotlib`, and `plasmapy`. No HelioAI, no API key.

This is the artefact you attach to a paper or share with a colleague.

In [ ]:
%%helioai
Export the complete St Patrick's Day storm analysis from this session as
a single standalone Python script.

The script should:
  1. Download the Wind and ACE data using speasy (no HelioAI import)
  2. Discard fill values using each variable's declared FILLVAL from the
     CDF metadata, not a hard-coded threshold -- Wind/SWE fills with
     99999.9, ACE with -1e31, and a fixed cutoff gets one of them wrong
  3. Reproduce the four-panel solar wind plot with shock annotation
  4. Print the Rankine-Hugoniot table
  5. Print the two-spacecraft timing result
  6. Have a single CONFIG block at the top (date range, parameter IDs,
     figure output path) so any user can adapt it to a different event
     by changing only that block

Add a header comment with: event name and date, the instrument
references (Lepping et al. 1995 for Wind/MFI, Ogilvie et al. 1995 for
Wind/SWE, Smith et al. 1998 for ACE/MAG), the Dst source (WDC Kyoto),
and the speasy version pinned.

Save the script as stpatrick_2015_standalone.py in the workspace.

---
## What you just did — and what took hours before HelioAI

| Task | Old workflow | HelioAI |
|:---|:---|:---|
| Find correct speasy IDs | 30 min CDAWeb browsing | ~30 s |
| Write download + plot code | 1–2 h Python/IDL | ~60 s |
| Compute Rankine-Hugoniot | 30 min manual formulas | ~60 s |
| Multi-spacecraft timing | 1 h coordinate geometry | ~90 s |
| Export reproducible script | 30 min refactoring | ~30 s |
| **Total** | **3–5 h** | **< 10 min** |

The agent made **no assumptions** about parameter IDs — it searched the 83 000-entry catalog
and returned canonical speasy paths, then reported the coverage it actually measured rather
than trusting that a download of *n* rows contains *n* measurements.

### The part worth keeping

Three of the traps in this notebook are not incidental, and each one silently produces a
plausible wrong answer:

- **Fill values are not NaN, and the sentinel is not universal.** Wind/SWE fills with
  99999.9, ACE with −1e31. A `|x| > 1e30` filter passes Wind's fill straight into your
  averages as a solar-wind speed. Read `FILLVAL` from the metadata. A blanket
  "reject ≥ 99999" rule is also wrong — OMNI carries a real proton temperature of 99093 K
  during the 2003 Halloween window.
- **`K0` means browse quality.** For this event `WI_K0_SWE/Np` has no fill values *and*
  single-point spikes to 166 cm⁻³, which put a false shock four hours early and a
  compression ratio of 1.1. Prefer `H*` science products.
- **A density jump is not a shock.** Require n, |B| and V to jump together, or the 12:51 UT
  compression structure will be misread as the shock.

### On the event that is not here

This notebook was written for the 2003 Halloween storm and moved after the coverage was
measured rather than assumed: ACE/SWEPAM is 100 % fill for that window, Wind/SWE 69–74 %,
OMNI 1-min 57 %, and SOHO/CELIAS clips its speed at 1019 km/s. The literature values that
were in this notebook's reference table — 70–100 cm⁻³, 1850–2000 km/s — cannot be
reproduced from any standard speasy product, and were attributed to an instrument that has
no data for those dates. Extreme events break instruments; that is worth knowing, and it is
why a first demonstration should not depend on one.

---

**References** — instrument and index sources only. Event-specific citations were removed
rather than reconstructed from memory; add your own.

- Lepping et al. (1995), *Space Sci. Rev.*, 71, 207 — Wind MFI magnetometer
- Ogilvie et al. (1995), *Space Sci. Rev.*, 71, 55 — Wind SWE solar wind experiment
- Smith et al. (1998), *Space Sci. Rev.*, 86, 613 — ACE magnetic field instrument
- [WDC for Geomagnetism, Kyoto](https://wdc.kugi.kyoto-u.ac.jp/dstdir/) — Dst index

**Links**
- [HelioAI on GitHub](https://github.com/erdoganfurkan/HelioAI)
- [speasy documentation](https://speasy.readthedocs.io)
- [PlasmaPy documentation](https://docs.plasmapy.org)
- [Wind project](https://wind.nasa.gov/)
- [ACE Science Center](https://www.srl.caltech.edu/ACE/ASC/)